# Distributed Trigger-Based Aggregation (Citus)

Compare two approaches for maintaining a confusion-matrix aggregation over
`pred_patch` × `patch`:

| Approach | Mechanism |
|---|---|
| **Trigger** | Statement-level `AFTER INSERT` trigger on `v4_pred_patch` incrementally upserts into `v4_confusion_matrix_partial` |
| **Materialized View** | `REFRESH MATERIALIZED VIEW` over a `GROUP BY` join |

Both tables (`v4_patch`, `v4_pred_patch`) are distributed on `id`.
`v4_confusion_matrix_partial` is distributed on `shard_id` (Citus shard ID).
Grid level: **l12** (finest, 4096×4096).

In [1]:
import psycopg2

In [2]:
DATABASE_URL = "dbname=citus user=citus password=citus host=localhost port=5439"
TABLE_PREFIX = "v4"
PATCH_TABLE = f"{TABLE_PREFIX}_patch"
PRED_PATCH_TABLE = f"{TABLE_PREFIX}_pred_patch"
CM_TABLE = f"{TABLE_PREFIX}_confusion_matrix_partial"
MATVIEW_NAME = f"{TABLE_PREFIX}_patch_label_agg_l12"

## 1 – Drop existing tables and views

In [3]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute(f"DROP MATERIALIZED VIEW IF EXISTS {MATVIEW_NAME} CASCADE;")
cur.execute(f"DROP TABLE IF EXISTS {CM_TABLE} CASCADE;")
cur.execute(f"DROP TABLE IF EXISTS {PRED_PATCH_TABLE} CASCADE;")
cur.execute(f"DROP TABLE IF EXISTS {PATCH_TABLE} CASCADE;")

conn.commit()
cur.close()
conn.close()

print(f"Dropped (if existed): {MATVIEW_NAME}, {CM_TABLE}, {PRED_PATCH_TABLE}, {PATCH_TABLE}")

Dropped (if existed): v4_patch_label_agg_l12, v4_confusion_matrix_partial, v4_pred_patch, v4_patch


## 2 – Create `v4_patch` (distributed on `id`)

In [4]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute(f"""
CREATE UNLOGGED TABLE {PATCH_TABLE}(
    id BIGINT NOT NULL,
    patch_uid integer NOT NULL,
    gt_label integer,
    event_ts timestamp with time zone NOT NULL DEFAULT now(),
    image_id integer,
    working_mag double precision,
    PRIMARY KEY(id)
);
""")
cur.execute(f"SELECT create_distributed_table('{PATCH_TABLE}', 'id');")

conn.commit()
cur.close()
conn.close()

print(f"Created and distributed {PATCH_TABLE} on 'id'.")

Created and distributed v4_patch on 'id'.


## 3 – Create `v4_pred_patch` (distributed on `id`)

In [5]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute(f"""
CREATE UNLOGGED TABLE {PRED_PATCH_TABLE}(
    id BIGINT NOT NULL,
    patch_uid bigint NOT NULL,
    embed_coords point,
    grid_cell_i int,
    grid_cell_j int,
    event_ts timestamp with time zone NOT NULL DEFAULT now(),
    pred_label integer,
    patch_coords point,
    PRIMARY KEY(id)
);

CREATE INDEX idx_{PRED_PATCH_TABLE}_grid_cells
    ON {PRED_PATCH_TABLE} (grid_cell_i, grid_cell_j);
""")
cur.execute(f"SELECT create_distributed_table('{PRED_PATCH_TABLE}', 'id');")

conn.commit()
cur.close()
conn.close()

print(f"Created and distributed {PRED_PATCH_TABLE} on 'id'.")

Created and distributed v4_pred_patch on 'id'.


## 4 – Create `v4_confusion_matrix_partial` (distributed on `shard_id`)

In [6]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute(f"""
CREATE TABLE {CM_TABLE} (
    shard_id    BIGINT NOT NULL,
    grid_i      INT NOT NULL,
    grid_j      INT NOT NULL,
    pred_label  INT NOT NULL,
    gt_label    INT NOT NULL,
    cnt         BIGINT DEFAULT 0,
    PRIMARY KEY (shard_id, grid_i, grid_j, pred_label, gt_label)
);
""")
# Colocate with pred_patch so shard-level triggers can do local joins
cur.execute(f"""
    SELECT create_distributed_table(
        '{CM_TABLE}', 'shard_id',
        colocate_with => '{PRED_PATCH_TABLE}'
    );
""")

conn.commit()
cur.close()
conn.close()

print(f"Created and distributed {CM_TABLE} on 'shard_id', colocated with {PRED_PATCH_TABLE}.")

Created and distributed v4_confusion_matrix_partial on 'shard_id', colocated with v4_pred_patch.


## 5 – Per-shard statement-level triggers

Citus doesn't support triggers on distributed tables directly, but we can
create them on individual **shard placements** using
`run_command_on_colocated_placements`.

Each shard of `v4_pred_patch` gets an `AFTER INSERT … FOR EACH STATEMENT`
trigger with a `REFERENCING NEW TABLE AS new_rows` transition table.
The trigger function:
1. Derives the colocated `v4_patch` shard name from `pg_dist_shard` metadata
2. JOINs `new_rows` with the local `v4_patch` shard to get `gt_label`
3. Pre-aggregates via `GROUP BY` and upserts into the colocated
   `v4_confusion_matrix_partial` shard

In [7]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# ---------------------------------------------------------------------------
# Step 1: Create the trigger function on the coordinator.
#         Citus 13 DDL propagation pushes it to workers automatically;
#         we also explicitly run it on workers as a safety net.
#
# NOTE: TG_ARGV[0] and TG_ARGV[1] arrive from run_command_on_colocated_placements
# as schema-qualified names (e.g. "public.v4_confusion_matrix_partial_102040").
# We must use %s (raw) rather than %I (identifier-quoted) so that the dot is
# parsed as a schema separator, not embedded in the identifier.
# ---------------------------------------------------------------------------
trigger_fn_sql = f"""
CREATE OR REPLACE FUNCTION update_cm_shard()
RETURNS TRIGGER LANGUAGE plpgsql AS $body$
DECLARE
    v_patch_shard text;
    v_pred_shardid bigint;
BEGIN
    -- Extract the numeric shard ID from the current shard table name
    -- e.g. 'v4_pred_patch_102008' -> 102008
    v_pred_shardid := (regexp_match(TG_TABLE_NAME, '(\d+)$'))[1]::bigint;

    -- Resolve the colocated v4_patch shard (same hash-range).
    -- Build as schema.table using the current trigger's schema (TG_TABLE_SCHEMA).
    SELECT TG_TABLE_SCHEMA || '.' || '{PATCH_TABLE}_' || shardid::text
      INTO v_patch_shard
    FROM pg_dist_shard
    WHERE logicalrelid = '{PATCH_TABLE}'::regclass
      AND shardminvalue = (
          SELECT shardminvalue FROM pg_dist_shard
          WHERE shardid = v_pred_shardid
      );

    -- TG_ARGV[0] = colocated confusion_matrix_partial shard (schema-qualified)
    -- Use %s (not %I) for schema-qualified names so the dot is a schema separator
    EXECUTE format(
        'INSERT INTO %s (shard_id, grid_i, grid_j, pred_label, gt_label, cnt) '
        'SELECT 0, nr.grid_cell_i, nr.grid_cell_j, nr.pred_label, '
        '       COALESCE(p.gt_label, -1), COUNT(*) '
        'FROM new_rows nr '
        'LEFT JOIN %s p ON nr.id = p.id '
        'GROUP BY nr.grid_cell_i, nr.grid_cell_j, nr.pred_label, p.gt_label '
        'ON CONFLICT (shard_id, grid_i, grid_j, pred_label, gt_label) '
        'DO UPDATE SET cnt = EXCLUDED.cnt + %s.cnt',
        TG_ARGV[0], v_patch_shard, TG_ARGV[0]
    );
    RETURN NULL;
END;
$body$;
"""

cur.execute(trigger_fn_sql)
conn.commit()

# Ensure function exists on workers (belt-and-suspenders)
cur.execute(f"SELECT run_command_on_workers($outer${trigger_fn_sql}$outer$);")
conn.commit()
print("Trigger function created on coordinator and workers.")

# ---------------------------------------------------------------------------
# Step 2: Create per-shard AFTER INSERT … FOR EACH STATEMENT triggers
#         using run_command_on_colocated_placements.
#         %s  = pred_patch shard name (used in ON %s)
#         %L  = cm shard name (passed as TG_ARGV[0])
# ---------------------------------------------------------------------------
cur.execute(f"""
SELECT run_command_on_colocated_placements(
    '{PRED_PATCH_TABLE}',
    '{CM_TABLE}',
    $cmd$
        CREATE TRIGGER trg_update_cm AFTER INSERT ON %s
        REFERENCING NEW TABLE AS new_rows
        FOR EACH STATEMENT
        EXECUTE FUNCTION update_cm_shard(%L)
    $cmd$
);
""")
conn.commit()

cur.close()
conn.close()

print("Per-shard statement-level triggers installed.")

Trigger function created on coordinator and workers.
Per-shard statement-level triggers installed.


## 6 – Data generation & insertion

Two-pass insertion:
1. **Phase A** – COPY all `v4_patch` rows (no trigger overhead).
2. **Phase B** – COPY all `v4_pred_patch` rows (trigger fires per-statement,
   incrementally updating `v4_confusion_matrix_partial`).

Both passes use identical RNG seeds per worker so IDs match across tables.

In [8]:
import numpy as np
import psycopg2
import time
from io import StringIO
from multiprocessing import Pool

# ======================
# CONFIG
# ======================
FINEST_LEVEL = 12
GRID_SIZE = 2 ** FINEST_LEVEL
NUM_LABELS = 5
TOTAL_ROWS = 1_000_000
BATCH_SIZE = 100_000
SPATIAL_CLUSTERS = 10
POINTS_PER_CLUSTER = (TOTAL_ROWS - 1_000) // SPATIAL_CLUSTERS
UNIFORM_POINTS = 1_000
WORKERS = 1


# ======================
# GRID FUNCTION (vectorized)
# ======================
def vectorized_point_to_cell(embed_x, embed_y, cell_size, level):
    scaled_size = cell_size / (2 ** level)
    i_arr = np.floor(embed_x / scaled_size).astype(np.int32)
    j_arr = np.floor(embed_y / scaled_size).astype(np.int32)
    return i_arr, j_arr


# ======================
# COPY HELPERS
# ======================
def build_patch_buffer(ids, patch_uids, gt_label):
    buf = StringIO()
    for k in range(len(ids)):
        buf.write(f"{ids[k]}\t{patch_uids[k]}\t{gt_label}\t\\N\t\\N\n")
    buf.seek(0)
    return buf


def build_pred_buffer(ids, patch_uids, ex, ey, grid_i, grid_j, pred_labels, px, py):
    buf = StringIO()
    for k in range(len(ids)):
        buf.write(
            f"{ids[k]}\t{patch_uids[k]}\t({ex[k]},{ey[k]})\t"
            f"{grid_i[k]}\t{grid_j[k]}\t{pred_labels[k]}\t({px[k]},{py[k]})\n"
        )
    buf.seek(0)
    return buf


# ======================
# BATCH GENERATORS (deterministic per worker)
# ======================
def generate_batches(worker_id):
    """Yield (ids, patch_uids, gt_label, ex, ey, grid_i, grid_j, pred_labels, px, py)
    for each batch belonging to this worker. Deterministic given worker_id."""
    local_rng = np.random.default_rng(2026 + worker_id)
    clusters_per_worker = SPATIAL_CLUSTERS // WORKERS

    # Each worker gets a full TOTAL_ROWS-wide ID range to avoid collisions.
    # Worker 0 generates extra UNIFORM_POINTS that push past TOTAL_ROWS//WORKERS.
    id_start = worker_id * TOTAL_ROWS + 1
    uid = id_start

    for cluster in range(worker_id * clusters_per_worker,
                         (worker_id + 1) * clusters_per_worker):
        mean_x = local_rng.uniform(GRID_SIZE * 0.1, GRID_SIZE * 0.9)
        mean_y = local_rng.uniform(GRID_SIZE * 0.1, GRID_SIZE * 0.9)
        std = local_rng.uniform(GRID_SIZE * 0.05, GRID_SIZE * 0.2)

        gt_label = cluster % NUM_LABELS
        pred_mode = (
            int(local_rng.integers(0, NUM_LABELS))
            if cluster % 2 == 0 else "random"
        )

        for start in range(0, POINTS_PER_CLUSTER, BATCH_SIZE):
            n = min(BATCH_SIZE, POINTS_PER_CLUSTER - start)

            ex = np.clip(local_rng.normal(mean_x, std, n), 0, GRID_SIZE)
            ey = np.clip(local_rng.normal(mean_y, std, n), 0, GRID_SIZE)

            ids = np.arange(uid, uid + n, dtype=np.int64)
            patch_uids = ids.copy()

            grid_i, grid_j = vectorized_point_to_cell(ex, ey, GRID_SIZE, FINEST_LEVEL)

            pred_labels = (
                local_rng.integers(0, NUM_LABELS, n, dtype=np.int32)
                if pred_mode == "random"
                else np.full(n, pred_mode, dtype=np.int32)
            )

            px = local_rng.uniform(0, 10_000, n)
            py = local_rng.uniform(0, 10_000, n)

            yield ids, patch_uids, gt_label, ex, ey, grid_i, grid_j, pred_labels, px, py
            uid += n

    # Uniform tail (worker 0 only)
    if worker_id == 0:
        n = UNIFORM_POINTS
        ex = local_rng.uniform(0, GRID_SIZE, n)
        ey = local_rng.uniform(0, GRID_SIZE, n)

        ids = np.arange(uid, uid + n, dtype=np.int64)
        patch_uids = ids.copy()
        grid_i, grid_j = vectorized_point_to_cell(ex, ey, GRID_SIZE, FINEST_LEVEL)
        pred_labels = local_rng.integers(0, NUM_LABELS, n, dtype=np.int32)
        px = local_rng.uniform(0, 10_000, n)
        py = local_rng.uniform(0, 10_000, n)

        yield ids, patch_uids, NUM_LABELS, ex, ey, grid_i, grid_j, pred_labels, px, py


# ======================
# WORKERS
# ======================
def worker_patch(worker_id):
    """Phase A: insert into v4_patch only."""
    conn = psycopg2.connect(DATABASE_URL)
    t0 = time.perf_counter()
    rows = 0
    try:
        for ids, patch_uids, gt_label, *_ in generate_batches(worker_id):
            buf = build_patch_buffer(ids, patch_uids, gt_label)
            cur = conn.cursor()
            cur.copy_from(
                buf, PATCH_TABLE,
                columns=("id", "patch_uid", "gt_label", "image_id", "working_mag"),
            )
            conn.commit()
            cur.close()
            rows += len(ids)
    finally:
        conn.close()
    elapsed = time.perf_counter() - t0
    print(f"[Patch W{worker_id}] {rows:,} rows in {elapsed:.2f}s "
          f"({rows/elapsed:,.0f} rows/sec)")
    return rows, elapsed


def worker_pred(worker_id):
    """Phase B: insert into v4_pred_patch (trigger fires per-statement)."""
    conn = psycopg2.connect(DATABASE_URL)
    t0 = time.perf_counter()
    rows = 0
    try:
        for ids, patch_uids, _gt, ex, ey, grid_i, grid_j, pred_labels, px, py in generate_batches(worker_id):
            buf = build_pred_buffer(ids, patch_uids, ex, ey, grid_i, grid_j, pred_labels, px, py)
            cur = conn.cursor()
            cur.copy_from(
                buf, PRED_PATCH_TABLE,
                columns=("id", "patch_uid", "embed_coords",
                         "grid_cell_i", "grid_cell_j",
                         "pred_label", "patch_coords"),
            )
            conn.commit()
            cur.close()
            rows += len(ids)
    finally:
        conn.close()
    elapsed = time.perf_counter() - t0
    print(f"[Pred  W{worker_id}] {rows:,} rows in {elapsed:.2f}s "
          f"({rows/elapsed:,.0f} rows/sec)")
    return rows, elapsed


print("Helpers defined.")

Helpers defined.


In [9]:
t_total = time.perf_counter()

# Phase A — patch rows (no trigger)
print("=== Phase A: Inserting v4_patch rows ===")
t0 = time.perf_counter()
with Pool(WORKERS) as p:
    patch_results = p.map(worker_patch, range(WORKERS))
patch_elapsed = time.perf_counter() - t0
patch_rows = sum(r[0] for r in patch_results)
print(f"Phase A done: {patch_rows:,} rows in {patch_elapsed:.2f}s\n")

# Phase B — pred_patch rows (trigger fires per-statement)
print("=== Phase B: Inserting v4_pred_patch rows (with trigger) ===")
t0 = time.perf_counter()
with Pool(WORKERS) as p:
    pred_results = p.map(worker_pred, range(WORKERS))
pred_elapsed = time.perf_counter() - t0
pred_rows = sum(r[0] for r in pred_results)
print(f"Phase B done: {pred_rows:,} rows in {pred_elapsed:.2f}s\n")

total_elapsed = time.perf_counter() - t_total
print(f"Total insertion: {patch_rows + pred_rows:,} rows in {total_elapsed:.2f}s")
print(f"  Patch throughput:     {patch_rows/patch_elapsed:,.0f} rows/sec")
print(f"  Pred+trigger throughput: {pred_rows/pred_elapsed:,.0f} rows/sec")

=== Phase A: Inserting v4_patch rows ===
[Patch W0] 1,000,000 rows in 6.75s (148,223 rows/sec)
Phase A done: 1,000,000 rows in 6.78s

=== Phase B: Inserting v4_pred_patch rows (with trigger) ===
[Pred  W0] 1,000,000 rows in 20.55s (48,665 rows/sec)
Phase B done: 1,000,000 rows in 20.58s

Total insertion: 2,000,000 rows in 27.36s
  Patch throughput:     147,423 rows/sec
  Pred+trigger throughput: 48,596 rows/sec


## 7 – Quick sanity check on the trigger table

In [10]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute(f"SELECT COUNT(*) FROM {CM_TABLE};")
cm_rows = cur.fetchone()[0]
print(f"{CM_TABLE} rows: {cm_rows:,}")

cur.execute(f"SELECT SUM(cnt) FROM {CM_TABLE};")
cm_total = cur.fetchone()[0]
print(f"{CM_TABLE} SUM(cnt): {cm_total:,}")

cur.execute(f"SELECT * FROM {CM_TABLE} LIMIT 10;")
for row in cur.fetchall():
    print(row)

cur.close()
conn.close()

v4_confusion_matrix_partial rows: 997,451
v4_confusion_matrix_partial SUM(cnt): 1,000,000
(0, 1632, 2617, 0, 0, 1)
(0, 986, 2663, 0, 0, 1)
(0, 1074, 2799, 0, 0, 1)
(0, 1320, 2669, 0, 0, 1)
(0, 1116, 2346, 0, 0, 1)
(0, 395, 3138, 0, 0, 1)
(0, 1521, 2904, 0, 0, 1)
(0, 948, 2518, 0, 0, 1)
(0, 1522, 2206, 0, 0, 1)
(0, 1640, 2047, 0, 0, 1)


## 8 – Create & refresh materialized view (comparison baseline)

In [11]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Use COALESCE to match the trigger's handling of NULL gt_label
cur.execute(f"""
CREATE MATERIALIZED VIEW {MATVIEW_NAME} AS
SELECT
    pp.pred_label,
    COALESCE(p.gt_label, -1) AS gt_label,
    pp.grid_cell_i,
    pp.grid_cell_j,
    COUNT(*) AS patch_count
FROM {PRED_PATCH_TABLE} pp
LEFT JOIN {PATCH_TABLE} p ON pp.id = p.id
GROUP BY pp.pred_label, COALESCE(p.gt_label, -1), pp.grid_cell_i, pp.grid_cell_j;
""")

cur.execute(f"""
CREATE INDEX idx_{MATVIEW_NAME}_grid
    ON {MATVIEW_NAME} (grid_cell_i, grid_cell_j);
""")

conn.commit()
print(f"Created materialized view {MATVIEW_NAME}.")

# Time a REFRESH to measure the refresh-from-stale cost
t0 = time.perf_counter()
cur.execute(f"REFRESH MATERIALIZED VIEW {MATVIEW_NAME};")
conn.commit()
matview_refresh_time = time.perf_counter() - t0
print(f"Refreshed {MATVIEW_NAME} in {matview_refresh_time:.3f}s")

cur.close()
conn.close()

Created materialized view v4_patch_label_agg_l12.
Refreshed v4_patch_label_agg_l12 in 2.430s


## 9 – Benchmark: query `v4_confusion_matrix_partial`

In [12]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

query_cm = f"""
    SELECT gt_label, pred_label, SUM(cnt) AS total_count
    FROM {CM_TABLE}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
"""

N_RUNS = 10
times_cm = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    cur.execute(query_cm)
    rows_cm = cur.fetchall()
    t1 = time.perf_counter()
    times_cm.append(t1 - t0)

cur.close()
conn.close()

times_cm_ms = [t * 1000 for t in times_cm]
print(f"=== {CM_TABLE} ===")
print(f"Rows returned: {len(rows_cm)}")
print(f"Runs: {N_RUNS}")
print(f"Min:    {min(times_cm_ms):.2f} ms")
print(f"Max:    {max(times_cm_ms):.2f} ms")
print(f"Mean:   {sum(times_cm_ms)/len(times_cm_ms):.2f} ms")
print(f"Median: {sorted(times_cm_ms)[N_RUNS//2]:.2f} ms")
print("\nSample results (first 10 rows):")
for row in rows_cm[:10]:
    print(f"  gt={row[0]}, pred={row[1]}, total={row[2]}")

=== v4_confusion_matrix_partial ===
Rows returned: 30
Runs: 10
Min:    28.86 ms
Max:    65.02 ms
Mean:   39.08 ms
Median: 33.05 ms

Sample results (first 10 rows):
  gt=0, pred=0, total=119987
  gt=0, pred=1, total=19840
  gt=0, pred=2, total=20077
  gt=0, pred=3, total=20065
  gt=0, pred=4, total=19831
  gt=1, pred=0, total=19987
  gt=1, pred=1, total=20041
  gt=1, pred=2, total=20111
  gt=1, pred=3, total=19730
  gt=1, pred=4, total=119931


## 10 – Benchmark: query materialized view

In [13]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

query_mv = f"""
    SELECT gt_label, pred_label, SUM(patch_count) AS total_count
    FROM {MATVIEW_NAME}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
"""

N_RUNS = 10
times_mv = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    cur.execute(query_mv)
    rows_mv = cur.fetchall()
    t1 = time.perf_counter()
    times_mv.append(t1 - t0)

cur.close()
conn.close()

times_mv_ms = [t * 1000 for t in times_mv]
print(f"=== {MATVIEW_NAME} ===")
print(f"Rows returned: {len(rows_mv)}")
print(f"Runs: {N_RUNS}")
print(f"Min:    {min(times_mv_ms):.2f} ms")
print(f"Max:    {max(times_mv_ms):.2f} ms")
print(f"Mean:   {sum(times_mv_ms)/len(times_mv_ms):.2f} ms")
print(f"Median: {sorted(times_mv_ms)[N_RUNS//2]:.2f} ms")
print("\nSample results (first 10 rows):")
for row in rows_mv[:10]:
    print(f"  gt={row[0]}, pred={row[1]}, total={row[2]}")

=== v4_patch_label_agg_l12 ===
Rows returned: 30
Runs: 10
Min:    94.80 ms
Max:    99.61 ms
Mean:   95.82 ms
Median: 95.37 ms

Sample results (first 10 rows):
  gt=0, pred=0, total=119987
  gt=0, pred=1, total=19840
  gt=0, pred=2, total=20077
  gt=0, pred=3, total=20065
  gt=0, pred=4, total=19831
  gt=1, pred=0, total=19987
  gt=1, pred=1, total=20041
  gt=1, pred=2, total=20111
  gt=1, pred=3, total=19730
  gt=1, pred=4, total=119931


## 11 – Correctness check

Compare the `(gt_label, pred_label) → total_count` from both approaches.
They should match exactly.

In [14]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Trigger-based totals
cur.execute(f"""
    SELECT gt_label, pred_label, SUM(cnt)
    FROM {CM_TABLE}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
""")
cm_totals = {(r[0], r[1]): r[2] for r in cur.fetchall()}

# Matview totals
cur.execute(f"""
    SELECT gt_label, pred_label, SUM(patch_count)
    FROM {MATVIEW_NAME}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
""")
mv_totals = {(r[0], r[1]): r[2] for r in cur.fetchall()}

cur.close()
conn.close()

# Compare
all_keys = sorted(set(cm_totals) | set(mv_totals))
mismatches = 0
for key in all_keys:
    cm_val = cm_totals.get(key, 0)
    mv_val = mv_totals.get(key, 0)
    if cm_val != mv_val:
        mismatches += 1
        print(f"  MISMATCH gt={key[0]}, pred={key[1]}: trigger={cm_val}, matview={mv_val}")

if mismatches == 0:
    print(f"All {len(all_keys)} (gt_label, pred_label) pairs match.")
else:
    print(f"\n{mismatches} mismatches out of {len(all_keys)} pairs.")

All 30 (gt_label, pred_label) pairs match.


## 12 – Summary

In [15]:
median_cm = sorted(times_cm_ms)[N_RUNS // 2]
median_mv = sorted(times_mv_ms)[N_RUNS // 2]

print(f"{'Metric':<35} {'Trigger':>12} {'Matview':>12}")
print("-" * 61)
print(f"{'Patch insert time (s)':<35} {patch_elapsed:>12.2f} {patch_elapsed:>12.2f}")
print(f"{'Pred insert time (s)':<35} {pred_elapsed:>12.2f} {'N/A':>12}")
print(f"{'Matview refresh time (s)':<35} {'N/A':>12} {matview_refresh_time:>12.3f}")
print(f"{'Query latency median (ms)':<35} {median_cm:>12.2f} {median_mv:>12.2f}")
print(f"{'(gt,pred) pairs returned':<35} {len(rows_cm):>12} {len(rows_mv):>12}")
print(f"{'Correctness mismatches':<35} {mismatches:>12} {'—':>12}")

Metric                                   Trigger      Matview
-------------------------------------------------------------
Patch insert time (s)                       6.78         6.78
Pred insert time (s)                       20.58          N/A
Matview refresh time (s)                     N/A        2.430
Query latency median (ms)                  33.05        95.37
(gt,pred) pairs returned                      30           30
Correctness mismatches                         0            —
